# BMEI Bioinputs: Reference Data for Nogales

Data-backed context for the BMEI microbial inoculation program at Finca Nogales,
Bruselas, Huila. Three bioinput formulations (biofertilizer, biological fungicide,
biological insecticide) using 20+ microbial species.

This notebook pulls together:
- **Climate data** (NASA POWER) — temperature, humidity, and rainfall patterns at Bruselas
- **Soil context** — literature-based soil profile for volcanic Andean coffee soils
- **Lipopeptide chemistry** (PubChem) — molecular properties of the antifungal compounds Bacillus produces
- **Organism inventory** — the 20+ species across 3 BMEI formulations
- **Growth dynamics** — modeled Bacillus growth curves at different temperatures
- **Disease pressure calendar** — when leaf rust and other diseases are highest risk

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook"

# Load scraped data
climate = pd.read_csv("../data/raw/bmei_bruselas_climate.csv")
lipo = pd.read_csv("../data/raw/bmei_lipopeptide_properties.csv")

print(f"Climate: {len(climate)} monthly records ({climate['year'].min()}\u2013{climate['year'].max()})")
print(f"Lipopeptides: {len(lipo)} compounds")

---

## 1. The BMEI Organism Inventory

Three formulations, 20+ species. Each organism has a specific job — nitrogen
fixation, phosphorus solubilization, fungal pathogen suppression, or insect
biocontrol. The chart below maps every organism to its formulation and function.

> **Source:** BMEI program documentation (Manual de BioM-E Programa Cafe especial,
> DESARROLLO BIOTECNOLOGICO BMEI CAFES ESPECIALES presentation, structured notes)
> provided by Weimar Obando Gómez / Weimagro SAS.

In [ ]:
# BMEI organism inventory (from BMEI documentation)
organisms = pd.DataFrame([
    # BioM-1: Biofertility
    {"organism": "Azospirillum brasilense", "formulation": "BioM-1", "type": "Bacterium",
     "function": "Nitrogen fixation", "detail": "Also produces auxins (root growth hormones)"},
    {"organism": "Azotobacter chroococcum", "formulation": "BioM-1", "type": "Bacterium",
     "function": "Nitrogen fixation", "detail": "Free-living N fixer, increases organic matter"},
    {"organism": "Pseudomonas fluorescens", "formulation": "BioM-1", "type": "Bacterium",
     "function": "P solubilization", "detail": "Also suppresses soilborne diseases"},
    {"organism": "Bacillus pumilus", "formulation": "BioM-1", "type": "Bacterium",
     "function": "Growth promotion", "detail": "Stress tolerance, ISR induction"},
    {"organism": "Bacillus megaterium", "formulation": "BioM-1", "type": "Bacterium",
     "function": "P solubilization", "detail": "Solubilizes insoluble phosphate minerals"},
    {"organism": "Penicillium janthinellum", "formulation": "BioM-1", "type": "Fungus",
     "function": "P solubilization", "detail": "Dissolves rock phosphate via organic acids"},
    {"organism": "Lactobacillus acidophilus", "formulation": "BioM-1", "type": "Bacterium",
     "function": "Fermentation", "detail": "Lactic acid production, pathogen suppression"},
    {"organism": "Saccharomyces cerevisiae", "formulation": "BioM-1", "type": "Yeast",
     "function": "Fermentation", "detail": "Rapid sugar consumption, B-vitamin production"},
    # BioM-2: Disease control
    {"organism": "Bacillus amyloliquefaciens QST713", "formulation": "BioM-2", "type": "Bacterium",
     "function": "Antifungal", "detail": "Produces iturins, surfactins, fengycins (Serenade\u00ae)"},
    {"organism": "Bacillus subtilis DSM 5750", "formulation": "BioM-2", "type": "Bacterium",
     "function": "Antifungal", "detail": "Biofilm formation, broad antibiotic production (Alicerce\u00ae)"},
    {"organism": "Bacillus licheniformis DSM 5749", "formulation": "BioM-2", "type": "Bacterium",
     "function": "Antifungal", "detail": "Extracellular enzymes, organic matter degradation (Alicerce\u00ae)"},
    # BioM-3: Insect control
    {"organism": "Metarhizium anisopliae", "formulation": "BioM-3", "type": "Fungus",
     "function": "Entomopathogen", "detail": "Kills thrips, mites via contact infection"},
    {"organism": "Beauveria bassiana", "formulation": "BioM-3", "type": "Fungus",
     "function": "Entomopathogen", "detail": "Primary biocontrol for broca (coffee berry borer)"},
    {"organism": "Bacillus thuringiensis var. kurstaki", "formulation": "BioM-3", "type": "Bacterium",
     "function": "Insecticide", "detail": "Cry toxins against defoliating larvae"},
    {"organism": "Bacillus popilliae", "formulation": "BioM-3", "type": "Bacterium",
     "function": "Insecticide", "detail": "Milky spore disease targets beetle larvae (broca)"},
    {"organism": "Lecanicillium lecanii", "formulation": "BioM-3", "type": "Fungus",
     "function": "Entomopathogen", "detail": "Whiteflies, aphids, mites"},
    {"organism": "Paecilomyces fumosoroseus", "formulation": "BioM-3", "type": "Fungus",
     "function": "Entomopathogen", "detail": "Broad-spectrum entomopathogen"},
    {"organism": "Methylobacterium symbioticum", "formulation": "BioM-3", "type": "Bacterium",
     "function": "Nitrogen fixation", "detail": "Methylotrophic N fixer, leaf surface colonizer"},
])

# Color by function category
function_colors = {
    "Nitrogen fixation": "#2ecc71",
    "P solubilization": "#3498db",
    "Growth promotion": "#1abc9c",
    "Fermentation": "#f39c12",
    "Antifungal": "#e74c3c",
    "Entomopathogen": "#9b59b6",
    "Insecticide": "#8e44ad",
}

fig = px.scatter(
    organisms,
    x="formulation",
    y="organism",
    color="function",
    symbol="type",
    size=[18] * len(organisms),
    hover_data=["detail"],
    title="BMEI Organism Inventory: 18 Species Across 3 Formulations",
    labels={"formulation": "", "organism": "", "function": "Function", "type": "Organism Type"},
    color_discrete_map=function_colors,
    symbol_map={"Bacterium": "circle", "Fungus": "diamond", "Yeast": "square"},
)
fig.update_layout(
    height=650,
    yaxis=dict(categoryorder="array",
               categoryarray=organisms.sort_values(["formulation", "function"])["organism"].tolist()),
    xaxis=dict(side="top"),
    legend=dict(orientation="h", yanchor="bottom", y=-0.15),
)
fig.update_traces(marker=dict(line=dict(width=1, color="#333")))
fig.show()

print(f"\nTotal organisms: {len(organisms)}")
print(f"  BioM-1 (Biofertility): {len(organisms[organisms['formulation'] == 'BioM-1'])} species")
print(f"  BioM-2 (Disease control): {len(organisms[organisms['formulation'] == 'BioM-2'])} species")
print(f"  BioM-3 (Insect control): {len(organisms[organisms['formulation'] == 'BioM-3'])} species")

**BioM-1** (biofertility) is the most diverse — 8 organisms handling nitrogen fixation,
phosphorus solubilization, and organic matter decomposition. These address the core
nutrient limitations of acidic Andean coffee soils.

**BioM-2** (disease control) is focused — 3 Bacillus species that produce lipopeptide
antibiotics (surfactin, iturin, fengycin). These directly combat coffee leaf rust,
anthracnose, and other fungal pathogens.

**BioM-3** (insect control) combines 4 entomopathogenic fungi with 2 Bacillus insecticide
strains. The primary target is broca (coffee berry borer), the #1 insect pest in coffee.

---

## 2. Bruselas Climate — What the Tanks and Diseases Face

Monthly temperature, humidity, and rainfall at Bruselas (1.85°N, 76.08°W, ~1800m)
from NASA POWER (2018–2025). This data matters for:
- **Tank management** — microbial growth rates are temperature-dependent
- **Disease pressure** — leaf rust thrives at 20–25°C with high humidity
- **Application timing** — when to apply BioM-2 (fungicide) and BioM-3 (insecticide)

> **Source:** [NASA POWER](https://power.larc.nasa.gov/) — Prediction of Worldwide
> Energy Resources, monthly point data via
> [`/api/temporal/monthly/point`](https://power.larc.nasa.gov/docs/services/api/).
> Parameters: T2M, T2M_MAX, T2M_MIN, T2M_RANGE, RH2M, PRECTOTCORR, ALLSKY_SFC_SW_DWN.
> Coordinates: 1.85°N, 76.08°W. No API key required.

In [ ]:
# Pivot climate data to monthly averages across years
monthly_avg = (
    climate.groupby(["month", "parameter"])["value"]
    .mean()
    .reset_index()
)

month_names = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
               7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
monthly_avg["month_name"] = monthly_avg["month"].map(month_names)

# Temperature subplot
temp_params = ["T2M", "T2M_MAX", "T2M_MIN"]
temp = monthly_avg[monthly_avg["parameter"].isin(temp_params)].pivot(
    index="month", columns="parameter", values="value"
).reset_index()
temp["month_name"] = temp["month"].map(month_names)

# Rainfall
rain = monthly_avg[monthly_avg["parameter"] == "PRECTOTCORR"].copy()
rain["monthly_mm"] = rain["value"] * 30  # mm/day → mm/month approx

# Humidity
humidity = monthly_avg[monthly_avg["parameter"] == "RH2M"].copy()

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        "Temperature (\u00b0C)",
        "Rainfall (mm/month)",
        "Relative Humidity (%)"
    ),
    vertical_spacing=0.1,
    shared_xaxes=True,
)

# Temperature band
fig.add_trace(go.Scatter(
    x=temp["month_name"], y=temp["T2M_MAX"],
    name="Max Temp", line=dict(color="#e74c3c", dash="dot"),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=temp["month_name"], y=temp["T2M"],
    name="Mean Temp", line=dict(color="#e67e22", width=3),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=temp["month_name"], y=temp["T2M_MIN"],
    name="Min Temp", line=dict(color="#3498db", dash="dot"),
), row=1, col=1)

# Leaf rust optimal zone (20-25°C)
fig.add_hrect(y0=20, y1=25, fillcolor="#e74c3c", opacity=0.1,
              annotation_text="Leaf rust optimal (20\u201325\u00b0C)",
              annotation_position="top right",
              row=1, col=1)

# Bacillus optimal zone (28-37°C)
fig.add_hrect(y0=28, y1=37, fillcolor="#2ecc71", opacity=0.1,
              annotation_text="Bacillus optimal (28\u201337\u00b0C)",
              annotation_position="top right",
              row=1, col=1)

# Rainfall
fig.add_trace(go.Bar(
    x=rain["month_name"], y=rain["monthly_mm"],
    name="Rainfall", marker_color="#3498db",
), row=2, col=1)

# Humidity
fig.add_trace(go.Scatter(
    x=humidity["month_name"], y=humidity["value"],
    name="Humidity", line=dict(color="#1abc9c", width=3),
    fill="tozeroy", fillcolor="rgba(26,188,156,0.2)",
), row=3, col=1)

# Leaf rust humidity threshold
fig.add_hline(y=80, line_dash="dash", line_color="#e74c3c",
              annotation_text="Leaf rust humidity threshold (80%)",
              row=3, col=1)

fig.update_layout(
    height=750,
    title="Bruselas, Huila \u2014 Monthly Climate (2018\u20132025 avg)",
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.08),
)
fig.update_yaxes(title_text="\u00b0C", row=1, col=1)
fig.update_yaxes(title_text="mm", row=2, col=1)
fig.update_yaxes(title_text="%", row=3, col=1)
fig.show()

**Key observations:**

- **Temperature is remarkably stable** year-round (~17–19°C mean). This is typical of
  equatorial highland coffee zones — minimal seasonal variation, but ~10°C diurnal range.
- **Ambient temperature is well below Bacillus optimal growth** (28–37°C). The BMEI
  tanks will need heat input or insulation to maintain good growth rates. At 18°C,
  Bacillus doubling time is ~3–4x longer than at 30°C.
- **Leaf rust conditions are met year-round** at Bruselas: temperature stays in the
  20–25°C range (max temps), and humidity consistently exceeds 80%. This explains
  why BioM-2 applications need to be monthly, not seasonal.
- **Dry spell July–August** is the only period with lower disease pressure. This is also
  when flowering is triggered for the next primary harvest (April–June).

> Leaf rust optimal temperature range (20–25°C) from Avelino et al. (2015),
> [doi:10.1007/s10584-015-1462-1](https://doi.org/10.1007/s10584-015-1462-1).
> Bacillus optimal growth range from Schallmey et al. (2004),
> [doi:10.1139/w04-028](https://doi.org/10.1139/w04-028).

---

## 3. Soil Context — Why Biofertilizers Matter Here

Bruselas sits on volcanic Andean soils (Andisols). These soils have high organic
matter but are typically acidic (pH 4.5–5.5), which locks up phosphorus and limits
availability of calcium, magnesium, and potassium. This is exactly what BioM-1's
phosphorus-solubilizing organisms are designed to address.

> **Note:** Soil values below are **estimates** based on regional Andisol literature,
> not from an actual soil test at Nogales. A real soil analysis would confirm or
> correct these. [ISRIC SoilGrids](https://soilgrids.org/) data for these coordinates
> was unavailable at scrape time (503 error).
>
> Nutrient availability vs pH curves follow the general pattern established by
> Truog (1946) and refined in subsequent soil science literature. The sigmoid
> parameters are approximate — the shape and relative ordering are well-established,
> the exact curves are modeled.

In [ ]:
# Nutrient availability vs soil pH (classic agronomic relationship)
# Data based on Truog (1946) and subsequent literature
ph_range = np.linspace(4.0, 8.0, 200)

def availability_curve(ph, optimal_low, optimal_high, drop_rate=3.0):
    """Model nutrient availability as a function of pH."""
    mid = (optimal_low + optimal_high) / 2
    width = (optimal_high - optimal_low) / 2
    return 1.0 / (1.0 + np.exp(-drop_rate * (ph - (mid - width)))) * \
           1.0 / (1.0 + np.exp(drop_rate * (ph - (mid + width))))

nutrients = {
    "Nitrogen (N)": (5.5, 8.0, 2.5),
    "Phosphorus (P)": (6.0, 7.5, 4.0),
    "Potassium (K)": (5.5, 8.0, 2.5),
    "Calcium (Ca)": (6.0, 8.5, 3.0),
    "Magnesium (Mg)": (5.5, 8.0, 2.5),
    "Iron (Fe)": (4.0, 6.5, 3.0),
    "Manganese (Mn)": (4.0, 6.5, 3.0),
    "Aluminum (toxic)": (3.5, 5.5, 3.5),
}

fig = go.Figure()

colors = {
    "Nitrogen (N)": "#2ecc71", "Phosphorus (P)": "#3498db",
    "Potassium (K)": "#e67e22", "Calcium (Ca)": "#95a5a6",
    "Magnesium (Mg)": "#1abc9c", "Iron (Fe)": "#e74c3c",
    "Manganese (Mn)": "#9b59b6", "Aluminum (toxic)": "#c0392b",
}

for nutrient, (low, high, rate) in nutrients.items():
    avail = availability_curve(ph_range, low, high, rate)
    dash = "dash" if nutrient == "Aluminum (toxic)" else "solid"
    fig.add_trace(go.Scatter(
        x=ph_range, y=avail, name=nutrient,
        line=dict(color=colors[nutrient], width=2.5, dash=dash),
    ))

# Bruselas pH range
fig.add_vrect(x0=4.8, x1=5.5, fillcolor="#f39c12", opacity=0.15,
              annotation_text="Bruselas pH range",
              annotation_position="top")

# Coffee optimal pH
fig.add_vrect(x0=5.5, x1=6.5, fillcolor="#2ecc71", opacity=0.08,
              annotation_text="Coffee optimal",
              annotation_position="top")

fig.update_layout(
    title="Nutrient Availability vs Soil pH \u2014 Why BioM-1 Matters",
    xaxis_title="Soil pH",
    yaxis_title="Relative Availability",
    height=500,
    yaxis=dict(showticklabels=False),
    legend=dict(orientation="h", yanchor="bottom", y=-0.2),
)
fig.show()

print("At Bruselas pH (4.8\u20135.5):")
print("  \u2713 Iron and Manganese: highly available (can be toxic)")
print("  \u2713 Aluminum: available (toxic to roots, inhibits P uptake)")
print("  \u2717 Phosphorus: severely limited (locked up as Al/Fe phosphates)")
print("  \u2717 Calcium & Magnesium: reduced availability")
print("  ~ Nitrogen & Potassium: moderate availability")
print("\nBioM-1 addresses this directly: Pseudomonas, Bacillus megaterium, and")
print("Penicillium solubilize locked-up phosphorus via organic acid production.")

In [ ]:
# Bruselas soil profile (literature values for volcanic Andean coffee soils, Huila)
# Sources: IGAC, Cenicafe soil surveys for southern Huila
soil_profile = pd.DataFrame([
    {"property": "pH (H\u2082O)", "value": 5.1, "optimal_low": 5.5, "optimal_high": 6.5,
     "unit": "", "status": "Below optimal"},
    {"property": "Organic Matter", "value": 8.5, "optimal_low": 4.0, "optimal_high": 10.0,
     "unit": "%", "status": "Good (volcanic)"},
    {"property": "Available P (Bray II)", "value": 6.0, "optimal_low": 20.0, "optimal_high": 40.0,
     "unit": "mg/kg", "status": "Deficient"},
    {"property": "Exchangeable Ca", "value": 2.5, "optimal_low": 3.0, "optimal_high": 6.0,
     "unit": "cmol/kg", "status": "Low"},
    {"property": "Exchangeable Mg", "value": 0.8, "optimal_low": 1.0, "optimal_high": 3.0,
     "unit": "cmol/kg", "status": "Low"},
    {"property": "Exchangeable K", "value": 0.25, "optimal_low": 0.2, "optimal_high": 0.6,
     "unit": "cmol/kg", "status": "Adequate"},
    {"property": "CEC", "value": 18.0, "optimal_low": 12.0, "optimal_high": 25.0,
     "unit": "cmol/kg", "status": "Good"},
    {"property": "Al Saturation", "value": 25.0, "optimal_low": 0.0, "optimal_high": 15.0,
     "unit": "%", "status": "High (toxic)"},
])

status_colors = {
    "Below optimal": "#e74c3c",
    "Deficient": "#c0392b",
    "Low": "#e67e22",
    "Adequate": "#2ecc71",
    "Good": "#27ae60",
    "Good (volcanic)": "#27ae60",
    "High (toxic)": "#c0392b",
}

fig = go.Figure()

for _, row in soil_profile.iterrows():
    # Normalize value to percentage of optimal range for visualization
    mid_optimal = (row["optimal_low"] + row["optimal_high"]) / 2
    pct = min(row["value"] / mid_optimal * 100, 150) if mid_optimal > 0 else 100
    color = status_colors[row["status"]]
    label = f"{row['value']} {row['unit']}" if row['unit'] else f"{row['value']}"

    fig.add_trace(go.Bar(
        y=[row["property"]], x=[pct],
        orientation="h",
        marker_color=color,
        text=f"{label} ({row['status']})",
        textposition="outside",
        showlegend=False,
        hovertemplate=f"{row['property']}: {label}<br>Optimal: {row['optimal_low']}\u2013{row['optimal_high']} {row['unit']}<br>Status: {row['status']}<extra></extra>",
    ))

fig.add_vline(x=100, line_dash="dash", line_color="gray",
              annotation_text="Optimal midpoint")

fig.update_layout(
    title="Estimated Soil Profile \u2014 Bruselas, Huila (Volcanic Andisol)",
    xaxis_title="% of Optimal Range Midpoint",
    height=400,
    yaxis=dict(categoryorder="array",
               categoryarray=soil_profile["property"].tolist()[::-1]),
    xaxis=dict(range=[0, 170]),
)
fig.show()

print("\nNote: Values are literature estimates for volcanic Andean coffee soils")
print("in southern Huila. Actual Nogales soil analysis would confirm these.")
print("\nThe pattern is classic for Andisols: high organic matter and CEC (good),")
print("but low pH driving P deficiency, Al toxicity, and Ca/Mg limitations.")

---

## 4. Lipopeptide Antibiotics — What Bacillus Actually Produces

The BioM-2 formulation's antifungal power comes from three families of lipopeptide
antibiotics produced by Bacillus species. These are cyclic peptides with fatty acid
tails that insert into fungal cell membranes and destroy them.

> **Source:** Molecular properties from [PubChem](https://pubchem.ncbi.nlm.nih.gov)
> PUG REST API ([docs](https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest)).
> CIDs: [surfactin (443592)](https://pubchem.ncbi.nlm.nih.gov/compound/443592),
> [iturin A (6437066)](https://pubchem.ncbi.nlm.nih.gov/compound/6437066),
> [fengycin (101099318)](https://pubchem.ncbi.nlm.nih.gov/compound/101099318),
> [beauvericin (105014)](https://pubchem.ncbi.nlm.nih.gov/compound/105014).
> Coffee compound properties also from PubChem (see pharmacognosy notebook).

In [ ]:
# Lipopeptide properties from PubChem
lipo_display = lipo[["compound_name", "MolecularWeight", "XLogP", "TPSA",
                      "HBondDonorCount", "HBondAcceptorCount"]].copy()
lipo_display["compound_label"] = lipo_display["compound_name"].str.replace("_", " ").str.title()

# Add biological context
bio_context = {
    "surfactin": {"producer": "Bacillus subtilis", "mechanism": "Membrane disruption",
                  "targets": "Broad-spectrum antifungal + antibacterial",
                  "class": "Lipopeptide"},
    "iturin_a": {"producer": "B. amyloliquefaciens", "mechanism": "Membrane pore formation",
                 "targets": "Strong antifungal, hemolytic",
                 "class": "Lipopeptide"},
    "fengycin": {"producer": "B. subtilis", "mechanism": "Membrane disruption",
                 "targets": "Filamentous fungi (leaf rust, anthracnose)",
                 "class": "Lipopeptide"},
    "beauvericin": {"producer": "Beauveria bassiana", "mechanism": "Ion channel disruption",
                    "targets": "Insecticidal + antifungal",
                    "class": "Cyclodepsipeptide"},
}

for _, row in lipo_display.iterrows():
    ctx = bio_context.get(row["compound_name"], {})
    for key, val in ctx.items():
        lipo_display.loc[lipo_display["compound_name"] == row["compound_name"], key] = val

# Compare with coffee compounds (from pharmacognosy notebook)
import os
coffee_props = None
if os.path.exists("../data/raw/pubchem_properties.csv"):
    coffee_props = pd.read_csv("../data/raw/pubchem_properties.csv")
    coffee_props["compound_label"] = coffee_props["compound_name"].str.replace("_", " ").str.title()
    coffee_props["source"] = "Coffee compound"

lipo_display["source"] = "BMEI metabolite"

if coffee_props is not None:
    combined = pd.concat([
        lipo_display[["compound_label", "MolecularWeight", "XLogP", "TPSA", "source"]],
        coffee_props[["compound_label", "MolecularWeight", "XLogP", "TPSA", "source"]],
    ], ignore_index=True)
else:
    combined = lipo_display[["compound_label", "MolecularWeight", "XLogP", "TPSA", "source"]].copy()

fig = px.scatter(
    combined,
    x="MolecularWeight",
    y="XLogP",
    size="TPSA",
    color="source",
    text="compound_label",
    title="BMEI Metabolites vs Coffee Compounds \u2014 Molecular Properties (PubChem)",
    labels={"MolecularWeight": "Molecular Weight (Da)", "XLogP": "LogP (lipophilicity)",
            "TPSA": "Polar Surface Area", "source": ""},
    color_discrete_map={"BMEI metabolite": "#e74c3c", "Coffee compound": "#3498db"},
)
fig.update_traces(textposition="top center", marker=dict(line=dict(width=1, color="#333")))

# Lipinski's Rule of 5 boundaries
fig.add_hline(y=5, line_dash="dot", line_color="gray",
              annotation_text="Lipinski LogP limit")
fig.add_vline(x=500, line_dash="dot", line_color="gray",
              annotation_text="Lipinski MW limit")

fig.update_layout(height=500)
fig.show()

print("\nBMEI lipopeptides are LARGE molecules (600\u20131000+ Da) and very lipophilic (LogP 3\u20139).")
print("This is why they work as membrane-disrupting antifungals \u2014 they insert into")
print("lipid bilayers. They violate Lipinski's Rule of 5, meaning they don't behave")
print("like conventional drugs \u2014 they act locally at the cell membrane, not systemically.")
print("\nCoffee compounds (caffeine, CGA, trigonelline) are much smaller and more")
print("polar \u2014 they're absorbed into the bloodstream and act on human targets.")
print("Different pharmacology, different scale.")

In [ ]:
# Lipopeptide antifungal mechanism comparison
mechanisms = pd.DataFrame([
    {"compound": "Surfactin", "membrane_disruption": 9, "pore_formation": 3,
     "biosurfactant": 10, "hemolytic": 2, "antifungal_potency": 6},
    {"compound": "Iturin A", "membrane_disruption": 7, "pore_formation": 9,
     "biosurfactant": 4, "hemolytic": 8, "antifungal_potency": 9},
    {"compound": "Fengycin", "membrane_disruption": 8, "pore_formation": 5,
     "biosurfactant": 3, "hemolytic": 1, "antifungal_potency": 8},
])

categories = ["Membrane\nDisruption", "Pore\nFormation", "Biosurfactant\nActivity",
              "Hemolytic\n(side effect)", "Antifungal\nPotency"]

fig = go.Figure()
colors = {"Surfactin": "#e74c3c", "Iturin A": "#3498db", "Fengycin": "#2ecc71"}
fill_rgba = {"Surfactin": "rgba(231,76,60,0.1)",
             "Iturin A": "rgba(52,152,219,0.1)",
             "Fengycin": "rgba(46,204,113,0.1)"}

for _, row in mechanisms.iterrows():
    values = [row["membrane_disruption"], row["pore_formation"],
              row["biosurfactant"], row["hemolytic"], row["antifungal_potency"]]
    values.append(values[0])  # close the polygon

    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        name=row["compound"],
        line=dict(color=colors[row["compound"]], width=2),
        fill="toself",
        fillcolor=fill_rgba[row["compound"]],
        opacity=0.8,
    ))

fig.update_layout(
    title="Bacillus Lipopeptide Mechanisms \u2014 Complementary Antifungal Profiles",
    polar=dict(radialaxis=dict(visible=True, range=[0, 10])),
    height=450,
)
fig.show()

print("The three lipopeptide families are complementary:")
print("  \u2022 Surfactin: best biosurfactant (wets leaf surfaces for better coverage)")
print("  \u2022 Iturin A: strongest antifungal + pore formation (kills fungi directly)")
print("  \u2022 Fengycin: strong antifungal with minimal hemolytic side effects (safest)")
print("\nBioM-2 uses all three producers together \u2014 this is the logic behind")
print("combining B. amyloliquefaciens (iturins) + B. subtilis (surfactin + fengycin).")

---

## 5. Growth Dynamics — What to Expect in the Tanks

Bacillus growth follows a classic sigmoid curve: lag phase → exponential growth
→ stationary phase → decline. Temperature is the primary driver of growth rate.
At Bruselas (~18°C ambient), growth will be significantly slower than at the
optimal 30–37°C.

> **Note:** Growth curves below are **modeled**, not measured. They use a logistic
> growth model with literature-informed doubling times. The relative differences
> between temperatures are well-established; the absolute cell densities and
> timing are approximate. Actual tank measurements would replace these models.
>
> Bacillus doubling times at various temperatures: ~25–30 min at 37°C,
> ~45–60 min at 25°C (Schallmey et al., 2004). Extrapolation to 18°C
> is estimated. Tank pH and Brix curves are modeled from general
> Bacillus fermentation behavior — not from BMEI-specific data.

In [ ]:
# Modeled Bacillus growth curves at different temperatures
# Based on literature doubling times:
#   30-37°C: ~25-30 min (optimal)
#   25°C: ~45-60 min
#   20°C: ~90-120 min
#   18°C (Bruselas ambient): ~150-180 min

def logistic_growth(t, N0, K, r, lag):
    """Logistic growth model with lag phase."""
    t_eff = np.maximum(t - lag, 0)
    return K / (1 + ((K - N0) / N0) * np.exp(-r * t_eff))

hours = np.linspace(0, 72, 500)
N0 = 1e6  # initial CFU/mL
K = 1e9   # carrying capacity

# Growth rate from doubling time: r = ln(2) / doubling_time_hours
conditions = {
    "37\u00b0C (lab optimal)": {"r": np.log(2) / 0.5, "lag": 2, "color": "#e74c3c"},
    "30\u00b0C (good)": {"r": np.log(2) / 0.75, "lag": 3, "color": "#e67e22"},
    "25\u00b0C (suboptimal)": {"r": np.log(2) / 1.0, "lag": 5, "color": "#f39c12"},
    "18\u00b0C (Bruselas ambient)": {"r": np.log(2) / 2.5, "lag": 8, "color": "#3498db"},
}

fig = go.Figure()

for label, params in conditions.items():
    N = logistic_growth(hours, N0, K, params["r"], params["lag"])
    fig.add_trace(go.Scatter(
        x=hours, y=N, name=label,
        line=dict(color=params["color"], width=2.5),
    ))

# Mark BMEI batch phases
fig.add_vrect(x0=0, x1=3, fillcolor="gray", opacity=0.05,
              annotation_text="Inoculation", annotation_position="top left")
fig.add_vline(x=24, line_dash="dot", line_color="gray",
              annotation_text="Day 1")
fig.add_vline(x=48, line_dash="dot", line_color="gray",
              annotation_text="Day 2")
fig.add_vline(x=72, line_dash="dot", line_color="gray",
              annotation_text="Day 3 (batch end)")

fig.update_layout(
    title="Modeled Bacillus Growth at Different Temperatures",
    xaxis_title="Hours",
    yaxis_title="Cell Density (CFU/mL)",
    yaxis_type="log",
    height=500,
    yaxis=dict(range=[5.5, 9.5]),
    legend=dict(yanchor="bottom", y=0.02, xanchor="right", x=0.98),
)
fig.show()

print("At 18\u00b0C (Bruselas ambient), Bacillus reaches stationary phase around")
print("48\u201372 hours \u2014 matching the BMEI's 3-day fermentation cycle.")
print("\nAt 30\u201337\u00b0C, stationary phase is reached in <24 hours. If the tanks")
print("are insulated or heated, batch time could be shortened significantly.")
print("\nKey monitoring implication: at 18\u00b0C, pH drop and OD increase will be")
print("slower and more gradual \u2014 don't assume contamination just because")
print("growth looks slow compared to textbook curves.")

In [ ]:
# Tank monitoring indicators — what pH and Brix changes to expect
# Based on Bacillus fermentation literature

# Modeled pH curves (Bacillus produces organic acids → pH drops, then stabilizes)
def ph_curve(t, initial, final, rate, lag):
    t_eff = np.maximum(t - lag, 0)
    return final + (initial - final) * np.exp(-rate * t_eff)

# Modeled Brix curves (sugar depletion)
def brix_curve(t, initial, final, rate, lag):
    t_eff = np.maximum(t - lag, 0)
    return final + (initial - final) * np.exp(-rate * t_eff)

days = np.linspace(0, 11, 200)  # Full BMEI cycle: 0-3 fermentation, 3-11 stabilization

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Expected pH Change (BioM-1 tank)", "Expected Brix Change (sugar depletion)"),
    vertical_spacing=0.15,
    shared_xaxes=True,
)

# pH: starts ~6.5-7.0 (molasses medium), drops to ~4.5-5.0
ph_healthy = ph_curve(days * 24, initial=6.8, final=4.5, rate=0.02, lag=8)
ph_slow = ph_curve(days * 24, initial=6.8, final=5.2, rate=0.008, lag=16)
ph_contaminated = ph_curve(days * 24, initial=6.8, final=3.5, rate=0.04, lag=4)

fig.add_trace(go.Scatter(x=days, y=ph_healthy, name="Healthy growth",
                         line=dict(color="#2ecc71", width=2.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=days, y=ph_slow, name="Slow growth (cold)",
                         line=dict(color="#f39c12", width=2, dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=days, y=ph_contaminated, name="Possible contamination",
                         line=dict(color="#e74c3c", width=2, dash="dot")), row=1, col=1)

# Safe pH range
fig.add_hrect(y0=4.0, y1=5.5, fillcolor="#2ecc71", opacity=0.05, row=1, col=1)

# Brix: starts ~3-5° (molasses), drops to ~0.5-1.0° as sugars consumed
brix_healthy = brix_curve(days * 24, initial=4.5, final=0.8, rate=0.015, lag=8)
brix_slow = brix_curve(days * 24, initial=4.5, final=1.5, rate=0.006, lag=16)

fig.add_trace(go.Scatter(x=days, y=brix_healthy, name="Brix (healthy)",
                         line=dict(color="#3498db", width=2.5), showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=days, y=brix_slow, name="Brix (slow)",
                         line=dict(color="#e67e22", width=2, dash="dash"), showlegend=False), row=2, col=1)

# Mark day transitions
for d in [3, 7, 11]:
    fig.add_vline(x=d, line_dash="dot", line_color="gray", row=1, col=1)
    fig.add_vline(x=d, line_dash="dot", line_color="gray", row=2, col=1)

fig.add_annotation(x=3, y=6.8, text="End fermentation", showarrow=False,
                   yshift=10, row=1, col=1)
fig.add_annotation(x=11, y=6.8, text="Application ready", showarrow=False,
                   yshift=10, row=1, col=1)

fig.update_yaxes(title_text="pH", row=1, col=1)
fig.update_yaxes(title_text="\u00b0Brix", row=2, col=1)
fig.update_xaxes(title_text="Days", row=2, col=1)

fig.update_layout(
    height=550,
    title="Tank Monitoring \u2014 Expected pH and Brix Curves (BioM-1 Cycle)",
    legend=dict(orientation="h", yanchor="bottom", y=-0.15),
)
fig.show()

print("Monitoring protocol for each batch:")
print("  Day 0: Record initial pH (~6.5\u20137.0) and Brix (~3\u20135\u00b0)")
print("  Day 1: pH should start dropping, Brix declining")
print("  Day 3: pH ~4.5\u20135.5, Brix ~1\u20132\u00b0 = healthy fermentation complete")
print("  Day 7: pH stabilized, ready for electromagnetic/stabilization steps")
print("  Day 11: Application ready")
print("\nRed flags:")
print("  \u2022 pH drops below 3.5 \u2192 possible acid-producing contaminant")
print("  \u2022 pH doesn't drop at all after 48h \u2192 organisms not growing")
print("  \u2022 Off-odors (putrid, sulfur) \u2192 contamination")
print("  \u2022 Brix unchanged after 48h \u2192 no metabolic activity")

---

## 6. Disease Pressure Calendar

Combining temperature, humidity, and rainfall data to estimate when coffee
diseases are most likely to appear at Bruselas. This informs BioM-2
application timing.

> **Source:** Climate inputs from [NASA POWER](https://power.larc.nasa.gov/)
> (same data as Section 2). Risk scores are a **composite index** — not from
> a published model. The component thresholds are from established epidemiology:
> leaf rust (20–25°C, >80% RH: Avelino et al., 2015), broca flight during dry
> periods (Jaramillo et al., 2006). The weighted scoring formula and resulting
> risk rankings are this notebook's construction.

In [ ]:
# Build disease pressure index from climate data
monthly_pivot = monthly_avg.pivot(index="month", columns="parameter", values="value").reset_index()
monthly_pivot["month_name"] = monthly_pivot["month"].map(month_names)
monthly_pivot["rainfall_mm"] = monthly_pivot["PRECTOTCORR"] * 30

# Leaf rust (Hemileia vastatrix) risk factors:
# - Temperature 20-25°C (max temp at Bruselas)
# - Humidity > 80%
# - Leaf wetness (correlated with rainfall)
# - Spore germination requires 5+ hours of leaf wetness

def rust_risk(temp_max, humidity, rainfall_mm):
    """Composite leaf rust risk score (0-10)."""
    # Temperature component (peak at 22°C)
    temp_score = max(0, 1 - abs(temp_max - 22) / 5) * 10
    # Humidity component (threshold at 80%)
    humid_score = min(max((humidity - 70) / 15, 0), 1) * 10
    # Rainfall component (wetness proxy)
    rain_score = min(rainfall_mm / 150, 1) * 10
    return (temp_score * 0.3 + humid_score * 0.4 + rain_score * 0.3)

def cbd_risk(temp_mean, humidity, rainfall_mm):
    """Coffee berry disease risk (favors cooler, wetter conditions)."""
    temp_score = max(0, 1 - abs(temp_mean - 18) / 5) * 10
    humid_score = min(max((humidity - 75) / 15, 0), 1) * 10
    rain_score = min(rainfall_mm / 120, 1) * 10
    return (temp_score * 0.3 + humid_score * 0.3 + rain_score * 0.4)

def broca_risk(temp_mean, rainfall_mm):
    """Coffee berry borer risk (warm + dry conditions favor flight/dispersal)."""
    temp_score = min(max((temp_mean - 15) / 10, 0), 1) * 10
    # Broca disperses during dry periods, but infests during fruit development
    rain_score = max(0, 1 - rainfall_mm / 200) * 5 + min(rainfall_mm / 100, 1) * 5
    return (temp_score * 0.5 + rain_score * 0.5)

monthly_pivot["rust_risk"] = monthly_pivot.apply(
    lambda r: rust_risk(r["T2M_MAX"], r["RH2M"], r["rainfall_mm"]), axis=1)
monthly_pivot["cbd_risk"] = monthly_pivot.apply(
    lambda r: cbd_risk(r["T2M"], r["RH2M"], r["rainfall_mm"]), axis=1)
monthly_pivot["broca_risk"] = monthly_pivot.apply(
    lambda r: broca_risk(r["T2M"], r["rainfall_mm"]), axis=1)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=monthly_pivot["month_name"], y=monthly_pivot["rust_risk"],
    name="Leaf Rust (BioM-2)", marker_color="#e74c3c", opacity=0.8,
))
fig.add_trace(go.Bar(
    x=monthly_pivot["month_name"], y=monthly_pivot["cbd_risk"],
    name="CBD / Anthracnose (BioM-2)", marker_color="#e67e22", opacity=0.8,
))
fig.add_trace(go.Bar(
    x=monthly_pivot["month_name"], y=monthly_pivot["broca_risk"],
    name="Broca (BioM-3)", marker_color="#9b59b6", opacity=0.8,
))

# Harvest window
fig.add_vrect(x0=2.5, x1=5.5, fillcolor="#2ecc71", opacity=0.08,
              annotation_text="Primary harvest", annotation_position="top")

fig.update_layout(
    title="Disease & Pest Pressure Calendar \u2014 Bruselas, Huila",
    xaxis_title="",
    yaxis_title="Risk Score (0\u201310)",
    barmode="group",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=-0.15),
)
fig.show()

print("Application timing recommendations:")
print("  BioM-2 (fungicide): Monthly year-round, increase frequency Mar\u2013May")
print("    (peak humidity + rainfall = peak rust/CBD risk during harvest)")
print("  BioM-3 (insecticide): Focus Apr\u2013Jun during cherry development")
print("    (broca infests developing cherries)")
print("  BioM-1 (biofertilizer): 6x/year, timed with rainfall for soil incorporation")
print("    (avoid dry Jul\u2013Aug when application can't reach root zone)")

---

## 7. Inter-year Climate Variability

Year-to-year variation in rainfall and temperature affects both disease
pressure and harvest timing. Understanding the range helps set expectations
for tank management and application schedules.

> **Source:** [NASA POWER](https://power.larc.nasa.gov/) monthly precipitation
> (PRECTOTCORR, mm/day) for 1.85°N, 76.08°W, 2018–2025.

In [ ]:
# Year-to-year rainfall variability
rain_monthly = climate[climate["parameter"] == "PRECTOTCORR"].copy()
rain_monthly["mm_month"] = rain_monthly["value"] * 30
rain_monthly["month_name"] = rain_monthly["month"].map(month_names)

fig = px.box(
    rain_monthly,
    x="month_name",
    y="mm_month",
    title="Rainfall Variability at Bruselas (2018\u20132025)",
    labels={"mm_month": "Monthly Rainfall (mm)", "month_name": ""},
    color_discrete_sequence=["#3498db"],
    category_orders={"month_name": list(month_names.values())},
)

# Add individual year points
fig.update_traces(boxpoints="all", jitter=0.3, pointpos=-1.8)

# Dry threshold
fig.add_hline(y=60, line_dash="dash", line_color="#e67e22",
              annotation_text="Dry threshold (60 mm)")

fig.update_layout(height=400)
fig.show()

# Annual totals
annual = rain_monthly.groupby("year")["mm_month"].sum()
print(f"\nAnnual rainfall range: {annual.min():.0f}\u2013{annual.max():.0f} mm")
print(f"Mean: {annual.mean():.0f} mm/year")
print(f"\nDriest months (July\u2013August) vary from {rain_monthly[rain_monthly['month'].isin([7,8])]['mm_month'].min():.0f}")
print(f"to {rain_monthly[rain_monthly['month'].isin([7,8])]['mm_month'].max():.0f} mm \u2014 some years barely dry at all.")

---

## Summary — What This Gives Her

### Data-backed premises for the BMEI work

| Finding | Source | Implication for BMEI |
|---------|--------|---------------------|
| Bruselas ambient temp ~18°C | [NASA POWER](https://power.larc.nasa.gov/) | Tank growth 3–4x slower than lab optimal; 3-day batch cycle makes sense |
| Humidity >80% year-round | [NASA POWER](https://power.larc.nasa.gov/) | Leaf rust pressure is constant, not seasonal; monthly BioM-2 applications justified |
| Acidic soil (pH ~5.0) | Estimated (Andisol literature) | P locked up, Al toxic; BioM-1 P-solubilizers directly address this |
| Surfactin MW 1036 Da, LogP 8.8 | [PubChem CID 443592](https://pubchem.ncbi.nlm.nih.gov/compound/443592) | Large lipophilic molecules — act locally at membranes, not systemically |
| Iturin A strongest antifungal | Estimated (literature review) | Justifies B. amyloliquefaciens as the anchor of BioM-2 |
| Peak disease pressure Mar–May | [NASA POWER](https://power.larc.nasa.gov/) + estimated risk model | Coincides with harvest — BioM-2/3 applications most critical during picking |

### What she can measure to validate

1. **pH + Brix at Day 0, 1, 2, 3, 11** — compare actual curves to the models above
2. **Tank temperature** — how much does it deviate from ambient 18°C? Does aeration warm it?
3. **Visual indicators** — biofilm formation (surface), color change, foam production
4. **Soil pH before/after BioM-1 application** — does it shift toward optimal?
5. **Disease incidence** — rust severity rating on BioM-2 treated vs untreated parcels

### Data sources

| Source | Records | What it provides | Verified |
|--------|---------|------------------|----------|
| [NASA POWER](https://power.larc.nasa.gov/) | 726 | Monthly temperature, humidity, rainfall for Bruselas (2018–2025) | Scraped via API |
| [PubChem](https://pubchem.ncbi.nlm.nih.gov) | 4 | Molecular properties of Bacillus lipopeptides + beauvericin | Scraped via PUG REST |
| BMEI documentation | 18 organisms | Three formulations with species, functions, and application rates | From Weimar / Weimagro SAS docs |
| Soil profile | 8 properties | Estimated Andisol values for Bruselas | Literature estimates, not measured |
| Growth / pH / Brix curves | — | Expected tank dynamics at 18°C | Modeled, not measured |
| Disease risk scores | — | Monthly risk index for rust, CBD, broca | Composite index (this notebook) |
| Lipopeptide mechanisms radar | — | Relative comparison of surfactin, iturin, fengycin | Qualitative (this notebook) |

### References

- Avelino, J. et al. (2015). The coffee rust crises in Colombia and Central America (2008–2013). *Food Security*, 7, 303–321. [doi:10.1007/s12571-015-0446-9](https://doi.org/10.1007/s12571-015-0446-9)
- Jaramillo, J. et al. (2006). Some like it hot: The influence and implications of climate change on coffee berry borer. *PLoS ONE*, 4(8). [doi:10.1371/journal.pone.0006487](https://doi.org/10.1371/journal.pone.0006487)
- Schallmey, M. et al. (2004). Developments in the use of Bacillus species for industrial production. *Can. J. Microbiol.*, 50(1), 1–17. [doi:10.1139/w04-028](https://doi.org/10.1139/w04-028)
- Ongena, M. & Jacques, P. (2008). Bacillus lipopeptides: versatile weapons for plant disease biocontrol. *Trends Microbiol.*, 16(3), 115–125. [doi:10.1016/j.tim.2007.12.009](https://doi.org/10.1016/j.tim.2007.12.009)
- Truog, E. (1946). Soil reaction influence on availability of plant nutrients. *Soil Science Society of America Journal*, 11, 305–308.

*Note: DOIs and publication details above are from training data and should be verified. Some years or journal details may be approximate.*